# SHERA ML-S01 Pairwise Correction Baseline

This notebook is an exploratory front end for the reusable `dluxshera.ml` modules. It inspects a prepared Wave 1 dataset, builds/reuses ML split and pair artifacts, samples ordered image pairs, and runs the ML-S01-E00 tiny-overfit path when PyTorch is available.

## 1. Configuration

Set `PREPARED_ROOT` to a Wave 1 prepared dataset. The other artifact paths are intentionally local defaults and can point anywhere on scratch or a project results filesystem.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

from notebook_setup import setup_paths
REPO_ROOT = setup_paths()

PREPARED_ROOT = Path("../../Results/ml_data/PREP-V3-v1").resolve()
ARTIFACT_ROOT = Path("../../Results/ml_experiments").resolve()
SPLIT_REGISTRY_PATH = ARTIFACT_ROOT / "splits" / "SPLIT-ML-v1.json"
VALIDATION_PAIRS_DIR = ARTIFACT_ROOT / "pair_manifests" / "PAIR-EVAL-v1-validation"
RUN_DIR = ARTIFACT_ROOT / "ML-S01" / "ML-S01-E00" / "ML-S01-E00-R001"

HAVE_PREPARED = PREPARED_ROOT.exists()
print("Prepared root:", PREPARED_ROOT)
print("Prepared available:", HAVE_PREPARED)

## 2. Catalog Inspection

The ML catalog is sample-centric: one row is one prepared rendered image/state. It keeps compact arrays and group IDs, not a large list of JSON dictionaries.

In [ ]:
from dluxshera.ml import load_sample_catalog

catalog = None
if HAVE_PREPARED:
    catalog = load_sample_catalog(PREPARED_ROOT)
    print(json.dumps(catalog.summary(), indent=2))
    print("parameter labels:", catalog.parameter_labels[:8], "...")
else:
    print("Set PREPARED_ROOT to inspect real prepared data.")

## 3. Split Registry

`SPLIT-ML-v1` splits science states and nuisance realizations separately. Science grouping uses the prepared physical-state hash, so the same science vector cannot leak across train/validation/test through repeated nuisance or pair-grid contexts.

In [ ]:
from dluxshera.ml import generate_split_registry, load_split_registry, write_split_registry

split_registry = None
if catalog is not None:
    if SPLIT_REGISTRY_PATH.exists():
        split_registry = load_split_registry(SPLIT_REGISTRY_PATH, catalog=catalog)
    else:
        split_registry = generate_split_registry(catalog, seed=11)
        write_split_registry(SPLIT_REGISTRY_PATH, split_registry)
    print(json.dumps(split_registry.counts, indent=2))

## 4. Pair Family Inspection

ML-S01-E01 starts with `same_nuisance_different_science`: each ordered pair keeps registration fixed and predicts `target_delta_z = z_B - z_A`. The default policy also keeps samples in the same V3 pair-grid context when `same_pair_id=True`.

In [ ]:
from dluxshera.ml import PairPolicy, PairSampler

policy = PairPolicy(
    policy_id="ml_s01_e01_clean_same_pair_grid_v1",
    family_weights={"same_nuisance_different_science": 1.0},
    same_pair_id=True,
    min_fisher_distance=0.25,
    max_fisher_distance=4.0,
    include_reverse=True,
)
example_pair = None
if catalog is not None and split_registry is not None:
    sampler = PairSampler(catalog, split_registry, policy)
    example_pair = sampler.sample_pair(np.random.default_rng(0), science_split="train", nuisance_split="train")
    print(json.dumps(example_pair.to_dict(), indent=2)[:2000])

## 5. Pair Images

Display `A`, `B`, and `B-A` for one sampled pair. `A` is the current/reference/model state and `B` is the target/observation state.

In [ ]:
if catalog is not None and example_pair is not None:
    with catalog.image_reader(cache_size=2) as reader:
        image_a = reader.get(example_pair.sample_a_index)
        image_b = reader.get(example_pair.sample_b_index)
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    for ax, image, title in zip(axes, [image_a, image_b, image_b - image_a], ["A", "B", "B - A"]):
        im = ax.imshow(image, origin="lower", cmap="magma")
        ax.set_title(title)
        ax.set_axis_off()
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    print("target delta z:", np.asarray(example_pair.target_delta_z))
    print("target delta theta:", np.asarray(example_pair.target_delta_theta))

## 6. Shared Encoder

A shared encoder means the same CNN instance and weights process both images: `h_A = E(I_A)` and `h_B = E(I_B)`. Weight sharing keeps the embedding coordinate system common, so `h_B - h_A` is meaningful as a learned relative feature. `concat_diff = [h_A, h_B, h_B - h_A]` is the default because it keeps relative information while retaining absolute-state context that may matter for nonlinear corrections.

In [ ]:
try:
    import torch
    from dluxshera.ml.models import build_pairwise_correction_model, count_parameters
except ModuleNotFoundError as exc:
    torch = None
    print(exc)

model = None
if torch is not None and catalog is not None:
    model = build_pairwise_correction_model(
        catalog.science_dim,
        {"comparator": "concat_diff", "embedding_dim": 128, "adaptive_pool_shape": [4, 4]},
    )
    print(model)
    print("trainable parameters:", count_parameters(model))

## 7. Tensor And Embedding Shapes

Downsampling trades pixel resolution for larger receptive fields and lower memory. This baseline keeps a small pooled spatial grid before the embedding rather than collapsing immediately to one global average.

In [ ]:
if torch is not None and model is not None and example_pair is not None:
    x_a = torch.from_numpy(image_a.astype("float32")).unsqueeze(0).unsqueeze(0)
    x_b = torch.from_numpy(image_b.astype("float32")).unsqueeze(0).unsqueeze(0)
    pred, h_a, h_b = model(x_a, x_b, return_embeddings=True)
    print("input:", tuple(x_a.shape))
    print("embedding:", tuple(h_a.shape))
    print("prediction:", tuple(pred.shape))

## 8. Image Scaling And Noise

This regression baseline avoids per-image L2 normalization because that can erase flux and amplitude information. The default transform is one scalar intensity scale derived only from the training partition. Prepared images stay noiseless; optional photon/read noise is applied dynamically to `B` when enabled.

In [ ]:
from dluxshera.ml import NoiseConfig, fit_intensity_scaler

if catalog is not None and split_registry is not None:
    train_indices = catalog.indices_for_groups(
        science_groups=split_registry.science_groups("train"),
        nuisance_groups=split_registry.nuisance_groups("train"),
    )
    scaler = fit_intensity_scaler(catalog, train_indices, mode="global_max_abs", max_samples=64)
    noise = NoiseConfig(enabled=False, apply_to="observation")
    print("scaler:", scaler.to_dict())
    print("noise:", noise.to_dict())

## 9. Frozen Evaluation Manifest

Validation and test comparisons should use fixed pair manifests. The same `pair_record_id`, `sample_a_id`, and `sample_b_id` keys can later join CNN predictions to a local-linear physics baseline without changing the training loop.

In [ ]:
from dluxshera.ml import generate_frozen_pair_manifest, load_pair_manifest, write_pair_manifest

validation_pairs = None
if catalog is not None and split_registry is not None:
    if VALIDATION_PAIRS_DIR.exists():
        validation_pairs = load_pair_manifest(VALIDATION_PAIRS_DIR, catalog=catalog, split_registry=split_registry)
    else:
        validation_pairs = generate_frozen_pair_manifest(
            catalog, split_registry, policy=policy, split="validation", seed=101, pairs_per_slice=64
        )
        write_pair_manifest(VALIDATION_PAIRS_DIR, validation_pairs)
    print(json.dumps(validation_pairs.summary(), indent=2))

## 10. ML-S01-E00 Tiny Overfit

E00 is a pipeline sanity check, not a generalization result. It verifies image loading, ordered-pair target construction, shared-CNN gradients, optimizer updates, checkpointing, and metrics artifacts on a tiny noiseless run.

In [ ]:
if torch is not None and catalog is not None and split_registry is not None:
    from dluxshera.ml.training import default_s01_e00_config, resolve_device, train_pairwise_correction
    config = default_s01_e00_config()
    config["device"] = "auto"
    print("resolved device:", resolve_device(config["device"]))
    config["training"]["epochs"] = 3
    config["training"]["pairs_per_epoch"] = 64
    summary = train_pairwise_correction(
        config=config,
        prepared_root=PREPARED_ROOT,
        split_registry_path=SPLIT_REGISTRY_PATH,
        output_dir=RUN_DIR,
        overwrite=True,
    )
    print(json.dumps(summary, indent=2))
else:
    print("Install PyTorch and set PREPARED_ROOT before running E00.")

## 11. History And Metrics Plots

In [ ]:
if (RUN_DIR / "history.csv").exists():
    import pandas as pd
    history = pd.read_csv(RUN_DIR / "history.csv")
    ax = history.plot(x="epoch", y=["train_loss", "validation_loss"], marker="o")
    ax.set_ylabel("MSE")
    plt.show()
    metrics = json.loads((RUN_DIR / "metrics.json").read_text())
    print(json.dumps(metrics, indent=2)[:3000])

## 12. Predicted Vs True Corrections

In [ ]:
pred_path = RUN_DIR / "evaluation_predictions.npz"
if pred_path.exists():
    data = np.load(pred_path)
    y_true = data["y_true_z"]
    y_pred = data["y_pred_z"]
    n = min(y_true.shape[1], 6)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3), squeeze=False)
    for j, ax in enumerate(axes.ravel()):
        ax.scatter(y_true[:, j], y_pred[:, j], s=10)
        lo = min(y_true[:, j].min(), y_pred[:, j].min())
        hi = max(y_true[:, j].max(), y_pred[:, j].max())
        ax.plot([lo, hi], [lo, hi], color="black", linewidth=1)
        ax.set_title(catalog.parameter_labels[j] if catalog is not None else f"z[{j}]")
        ax.set_xlabel("true")
        ax.set_ylabel("pred")
    plt.tight_layout()

## 13. Reverse-Pair Consistency Examples

In [ ]:
if torch is not None and model is not None and example_pair is not None:
    with torch.no_grad():
        f_ab = model(x_a, x_b).detach().numpy()[0]
        f_ba = model(x_b, x_a).detach().numpy()[0]
    print("f(A,B) + f(B,A):", f_ab + f_ba)

## 14. Comparator Switch

In [ ]:
if torch is not None and catalog is not None:
    difference_model = build_pairwise_correction_model(
        catalog.science_dim,
        {"comparator": "difference", "embedding_dim": 128, "adaptive_pool_shape": [4, 4]},
    )
    print("difference comparator model parameters:", count_parameters(difference_model))

## 15. Load External Checkpoint

In [ ]:
CHECKPOINT_PATH = RUN_DIR / "checkpoint_best.pt"
if torch is not None and catalog is not None and CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    loaded_model = build_pairwise_correction_model(catalog.science_dim, checkpoint["config"].get("model"))
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()
    print("loaded epoch:", checkpoint["epoch"])